# Mini Project 1 — Analysis Notebook

**Your name: Ning Zhang** 

**Dataset:**

FY24 Historical Reservations Full.csv (Recreation.gov historical reservations export); this notebook uses the processed subset `data/processed/yellowstone_summer_2024.csv` (Yellowstone National Park, `inventorytype` = CAMPING, stay start dates in June–August 2024). Optional context: RIDB facility metadata from recreation.gov RIDB API when cited.

**Date:**


In [2]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

**Dataset:** Two federal datasets from the Recreation One Stop system. (1) RIDB API (`ridb.recreation.gov/api/v1`) — facility-level metadata for Yellowstone campgrounds including amenities, campsite attributes, and site types, accessed via REST API with a free API key. (2) Recreation.gov Historical Reservation CSV (`ridb.recreation.gov/download`) — filtered to Yellowstone National Park camping reservations (`parentlocation == "Yellowstone National Park"`, `inventorytype == "CAMPING"`) for summer 2024 (June–August), joined to the RIDB data on shared `facilityid`.

**Why this dataset:** Public land reservation systems are a high-stakes digital access interface, and understanding how visitors navigate booking at America's most iconic park connects directly to HCD questions about discoverability, equity, and whether design is concentrating demand or distributing it.

**Three analytical questions:**

1. Which Yellowstone campgrounds have the highest cancellation rates in summer 2024, and do those same facilities also show the shortest average booking lead times — suggesting frustrated demand rather than genuine disinterest?
2. Do campgrounds with more listed amenities (per RIDB API) generate higher reservation volume or longer average stays, or are simpler campgrounds equally in demand?
3. Which campgrounds are being reserved the furthest in advance in summer 2024, and what does that pattern suggest about perceived desirability across Yellowstone's facilities?

**What a practitioner would do with these findings:** A national park service UX or operations team could use these findings to redesign how campground information is surfaced in the reservation interface — prioritizing underbooked but comparable facilities to reduce overcrowding and improve access equity.

---
## Section 2 — Data Profile

Read the markdown, then run each code cell under **2.1**–**2.4**. Output is **text only** (no charts).


### 2.1 How many rows and columns?

**Reservations:** one row per historical booking; fixed width (35 fields). **RIDB:** row counts depend on which endpoints you stack (facilities under Yellowstone rec area 2988, then campsites per facility).


In [3]:
from pathlib import Path
import json
import os
import ssl
import urllib.request

import pandas as pd

csv_path = Path("data/processed/yellowstone_summer_2024.csv")
df = pd.read_csv(csv_path, low_memory=False)
n_rows, n_cols = df.shape
print("Reservations CSV:", n_rows, "rows,", n_cols, "columns")

def _load_ridb_key():
    for fname in (".env", "yellowstone.env"):
        path = Path(fname)
        if not path.is_file():
            continue
        for line in path.read_text(encoding="utf-8").splitlines():
            s = line.strip()
            if s.startswith("RIDB_API_KEY="):
                return s.split("=", 1)[1].strip().strip('"').strip("'")
    return os.environ.get("RIDB_API_KEY", "").strip()

def _ridb_facility_count():
    key = _load_ridb_key()
    if not key:
        return None
    url = "https://ridb.recreation.gov/api/v1/recareas/2988/facilities?limit=500"
    req = urllib.request.Request(
        url,
        headers={"accept": "application/json", "apikey": key},
        method="GET",
    )
    try:
        import certifi
        ctx = ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        ctx = ssl.create_default_context()
    with urllib.request.urlopen(req, context=ctx, timeout=60) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return len(data.get("RECDATA") or [])

ridb_n = _ridb_facility_count()
if ridb_n is None:
    print("RIDB: could not fetch (missing RIDB_API_KEY in .env / yellowstone.env or network error).")
else:
    print("RIDB facilities (rec area 2988, limit=500):", ridb_n, "records")


Reservations CSV: 15326 rows, 35 columns
RIDB facilities (rec area 2988, limit=500): 25 records


### 2.2 What does each column represent?

**Reservations:** mix of ids, hierarchy, facility/geo, timing, party/equipment, and payment fields (see `df.info()`). **RIDB:** nested JSON (addresses, media, campsites, attributes) keyed by `FacilityID`/`FacilityName`.


In [4]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path("data/processed/yellowstone_summer_2024.csv"), low_memory=False, nrows=8000)
print("Column dtypes (8k-row sample):")
print(df.dtypes.value_counts())
print("\nColumn names:")
print(list(df.columns))


Column dtypes (8k-row sample):
str        18
float64    11
int64       6
Name: count, dtype: int64

Column names:
['historicalreservationid', 'ordernumber', 'agency', 'orgid', 'codehierarchy', 'regioncode', 'regiondescription', 'parentlocationid', 'parentlocation', 'legacyfacilityid', 'park', 'sitetype', 'usetype', 'productid', 'inventorytype', 'facilityid', 'facilityzip', 'facilitystate', 'facilitylongitude', 'facilitylatitude', 'customerzip', 'tax', 'usefee', 'tranfee', 'attrfee', 'totalbeforetax', 'discount', 'totalpaid', 'startdate', 'enddate', 'orderdate', 'nights', 'numberofpeople', 'equipmentdescription', 'equipmentlength']


### 2.3 Data quality snapshot

**Reservations:** sparse optional fields, `nights` as text, heterogeneous datetimes. **RIDB:** nested lists need flattening; rare non-numeric facility ids in other extracts.


In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path("data/processed/yellowstone_summer_2024.csv"), low_memory=False)
miss = df.isna().mean().sort_values(ascending=False).head(15)
print("Top missingness (share of rows):")
print(miss.to_string())

nights_head = df["nights"].astype(str).str.extract(r"(\d+)", expand=False)
nd = pd.to_numeric(nights_head, errors="coerce")
print("\nLeading digit from `nights` text (value_counts, top 10):")
print(nd.dropna().astype(int).value_counts().head(10).to_string())


Top missingness (share of rows):
usefee                  1.000000
tranfee                 1.000000
attrfee                 1.000000
customerzip             0.187655
equipmentdescription    0.017226
sitetype                0.000065
usetype                 0.000065
equipmentlength         0.000065
enddate                 0.000065
numberofpeople          0.000065
totalpaid               0.000000
facilitylatitude        0.000000
totalbeforetax          0.000000
startdate               0.000000
orderdate               0.000000

Leading digit from `nights` text (value_counts, top 10):
nights
1     9268
2     2962
3     1568
4      715
5      338
6      145
7      129
8       54
14      44
9       35


### 2.4 Analysis focus

**Reservations:** `facilityid`, `orderdate`/`startdate`, `park`, `nights`, `numberofpeople`, `totalpaid`. **RIDB:** campsite/attribute payloads keyed by `FacilityID` for amenity depth and maps.


In [ ]:
from pathlib import Path
import pandas as pd

df = pd.read_csv(Path("data/processed/yellowstone_summer_2024.csv"), low_memory=False)
g = (
    df.groupby("park", observed=True)
    .agg(
        reservations=("historicalreservationid", "count"),
        median_paid=("totalpaid", "median"),
        mean_paid=("totalpaid", "mean"),
    )
    .sort_values("reservations", ascending=False)
)
print("Reservations by campground (focus columns preview):")
print(g.to_string())


In [26]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


In [27]:
# Summary statistics for numeric columns
df.describe()

,id,rating,helpful_votes
count,500.000000,500.000000,500.000000
mean,250.500000,3.946000,23.464000
std,144.481833,1.184013,13.766471
min,1.000000,1.000000,0.000000
25%,125.750000,3.000000,11.000000
50%,250.500000,4.000000,23.500000
75%,375.250000,5.000000,35.000000
max,500.000000,5.000000,47.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1 — Cancellation rates vs. booking lead time:** Which Yellowstone campgrounds have the highest cancellation rates in summer 2024, and do those same facilities also show the shortest average booking lead times (suggesting frustrated demand rather than genuine disinterest)?

**Definitions for this analysis:** The historical reservation extract does not include an official cancellation flag. We use a **cancellation proxy**: any row with **`totalpaid == 0`** is counted toward a campground’s **cancellation proxy rate** (% of rows). Some $0 rows may be comps, refunds, or data quirks—not necessarily cancellations. **Booking lead time** is **`startdate` minus `orderdate`**, expressed in **days** (positive values only: bookings made before the stay starts).

In [5]:
# Question 1 — cancellation proxy rate vs. median booking lead time by campground
from pathlib import Path

import pandas as pd
import plotly.express as px

df = pd.read_csv(Path("data/processed/yellowstone_summer_2024.csv"), low_memory=False)

# Parse times (UTC-normalized); coerce bad strings from exports
df["order_ts"] = pd.to_datetime(df["orderdate"], utc=True, errors="coerce")
df["start_ts"] = pd.to_datetime(df["startdate"], utc=True, errors="coerce")

# Lead time in days (drop invalid or negative lead times)
df["lead_time_days"] = (df["start_ts"] - df["order_ts"]).dt.total_seconds() / 86400.0
df = df[df["lead_time_days"].notna() & (df["lead_time_days"] >= 0)].copy()

# Cancellation proxy: no payment recorded (see Question 1 markdown for caveats)
tp = pd.to_numeric(df["totalpaid"], errors="coerce").fillna(0)
df["cancelled_proxy"] = tp == 0

by_park = (
    df.groupby("park", observed=True)
    .agg(
        cancellation_proxy_rate_pct=("cancelled_proxy", lambda s: 100.0 * s.mean()),
        median_lead_time_days=("lead_time_days", "median"),
        n_reservations=("historicalreservationid", "count"),
    )
    .reset_index()
    .sort_values("cancellation_proxy_rate_pct", ascending=False)
)

fig = px.scatter(
    by_park,
    x="cancellation_proxy_rate_pct",
    y="median_lead_time_days",
    text="park",
    size="n_reservations",
    hover_data=["n_reservations"],
    title=(
        "Yellowstone summer 2024 camping: cancellation proxy rate vs. "
        "median booking lead time by campground"
    ),
    labels={
        "cancellation_proxy_rate_pct": (
            "Cancellation proxy rate (% of reservations with $0 total paid)"
        ),
        "median_lead_time_days": "Median booking lead time (days)",
        "n_reservations": "Reservation count",
        "park": "Campground",
    },
)
fig.update_traces(textposition="top center")
fig.show()

print(by_park.to_string(index=False))


                                 park  cancellation_proxy_rate_pct  median_lead_time_days  n_reservations
                Lewis Lake Campground                     1.786827              79.319994            4813
              Slough Creek Campground                     0.889454              13.414473             787
Indian Creek Campground (Yellowstone)                     0.501672              40.010051            4186
     Mammoth Campground (Yellowstone)                     0.439855              13.295574            5229


**Interpretation:** With the $0 `totalpaid` proxy, **Lewis Lake Campground** has the **highest** cancellation proxy rate (~1.8%) but also the **longest** median booking lead time (~79 days), so this slice does **not** support a simple story that the “most cancelled” parks are the same as the “shortest lead time” parks. **Mammoth** and **Slough Creek** cluster at **low** proxy cancellation (~0.4–0.9%) and **shorter** median leads (~13 days), while **Indian Creek** sits in the middle on both. Next steps would validate the proxy against an explicit reservation status field (if available) and check refund or modification patterns that zero out `totalpaid` without a true cancellation.

**Question 2 — Amenities vs. reservation volume / length of stay:** Do campgrounds with more listed amenities (per RIDB) generate higher reservation volume or longer average stays, or are simpler campgrounds equally in demand?

**Definitions for this analysis:** Reservation **volume** is the **count of rows** in `yellowstone_summer_2024.csv` by `park`. **Average stay length** uses the **`nights`** field: we parse the **leading whole number** from strings like `"4 days"` and take the **mean per park** (`avg_nights`). RIDB does not give a single “amenity score” in one call without per-campsite attribute requests; here **`amenity_count`** means the **number of RIDB campsite records** returned for the campground’s `facilityid` (paginated `/facilities/{id}/campsites`), used as a **richness / reservable-inventory proxy** correlated with how much site-level detail RIDB holds for that facility.

In [6]:
# Question 2 — RIDB campsite inventory (proxy) vs. reservations and avg stay
from pathlib import Path
import json
import os
import ssl
import urllib.request

import pandas as pd
import plotly.express as px

df = pd.read_csv(Path("data/processed/yellowstone_summer_2024.csv"), low_memory=False)

# Avg stay: leading number from "nights" text (e.g. "4 days" -> 4)
nights_head = df["nights"].astype(str).str.extract(r"(\d+)", expand=False)
df["nights_num"] = pd.to_numeric(nights_head, errors="coerce")

res = (
    df.groupby("park", observed=True)
    .agg(
        n_reservations=("historicalreservationid", "count"),
        avg_nights=("nights_num", "mean"),
        facilityid=(
            "facilityid",
            lambda s: int(pd.Series(s.dropna()).astype(int).value_counts().index[0]),
        ),
    )
    .reset_index()
)


def _load_ridb_key():
    for fname in (".env", "yellowstone.env"):
        p = Path(fname)
        if not p.is_file():
            continue
        for line in p.read_text(encoding="utf-8").splitlines():
            s = line.strip()
            if s.startswith("RIDB_API_KEY="):
                return s.split("=", 1)[1].strip().strip('"').strip("'")
    return os.environ.get("RIDB_API_KEY", "").strip()


def _ridb_campsite_count(facility_id: int) -> int:
    """Paginated count of RIDB /facilities/{id}/campsites RECDATA rows."""
    key = _load_ridb_key()
    if not key:
        raise RuntimeError("Missing RIDB_API_KEY (.env or yellowstone.env)")
    try:
        import certifi
        ctx = ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        ctx = ssl.create_default_context()
    total = 0
    offset = 0
    limit = 500
    while True:
        url = f"https://ridb.recreation.gov/api/v1/facilities/{facility_id}/campsites?limit={limit}&offset={offset}"
        req = urllib.request.Request(
            url,
            headers={"accept": "application/json", "apikey": key},
            method="GET",
        )
        with urllib.request.urlopen(req, context=ctx, timeout=60) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
        batch = payload.get("RECDATA") or []
        total += len(batch)
        if len(batch) < limit:
            break
        offset += limit
    return total


df_summary = res.copy()
df_summary["amenity_count"] = [
    _ridb_campsite_count(int(fid)) for fid in df_summary["facilityid"]
]

fig = px.bar(
    df_summary.sort_values("amenity_count", ascending=False),
    x="park",
    y="amenity_count",
    color="avg_nights",
    title="Amenity Count per Campground (colored by Avg Stay Length)",
    labels={
        "amenity_count": "Number of Amenities",
        "avg_nights": "Avg Nights Stayed",
        "park": "Campground",
    },
)
fig.update_layout(xaxis_tickangle=-20, height=480)
fig.show()

print(df_summary.sort_values("amenity_count", ascending=False).to_string(index=False))


                                 park  n_reservations  avg_nights  facilityid  amenity_count
     Mammoth Campground (Yellowstone)            5435    1.750874      247571             87
                Lewis Lake Campground            4813    1.804239      259309             84
Indian Creek Campground (Yellowstone)            4186    1.981366      259304             73
              Slough Creek Campground             892    2.096413      259310             17


**Interpretation:** Using **RIDB campsite record counts** as `amenity_count`, **Mammoth** and **Lewis Lake** sit at the **high** end for both listed campsites and **reservation volume**, while **Slough Creek** has the **fewest** RIDB campsites yet still draws a **higher mean parsed nights** (~2.1) than the others (~1.8–2.0), so “more RIDB-listed sites” does not line up one-to-one with “longer stays” in this slice. **Indian Creek** is mid-pack on campsite count and volume. A fuller amenities analysis would pull **per-campsite attributes** from RIDB (more API calls) or merge a dedicated facility-amenity table if your instructor provides one.

**Question 3 — Booking lead time by campground:** Which campgrounds are being reserved the furthest in advance in summer 2024, and what does that pattern suggest about perceived desirability across Yellowstone's facilities?

**Definitions for this analysis:** Rows are the **summer 2024 camping** extract (`yellowstone_summer_2024.csv`). **Booking lead time** is **`startdate` minus `orderdate`**, in **whole days** (`lead_time_days`), using UTC-normalized parsing. Rows with **missing** timestamps or **negative** lead times (data artifacts or same-day edge cases) are **dropped** before plotting. A **box plot** compares the **full distribution** of lead times per `park` (median, spread, and outliers), not just the average.

In [7]:
# Question 3 — booking lead time distribution by campground (summer 2024)
from pathlib import Path

import pandas as pd
import plotly.express as px

df_summer = pd.read_csv(Path("data/processed/yellowstone_summer_2024.csv"), low_memory=False)
df_summer["order_ts"] = pd.to_datetime(df_summer["orderdate"], utc=True, errors="coerce")
df_summer["start_ts"] = pd.to_datetime(df_summer["startdate"], utc=True, errors="coerce")
df_summer["lead_time_days"] = (
    df_summer["start_ts"] - df_summer["order_ts"]
).dt.total_seconds() / 86400.0
df_summer = df_summer[
    df_summer["lead_time_days"].notna() & (df_summer["lead_time_days"] >= 0)
].copy()

fig = px.box(
    df_summer,
    x="park",
    y="lead_time_days",
    color="park",
    title="Booking Lead Time Distribution by Campground (Summer 2024)",
    labels={
        "lead_time_days": "Days Booked in Advance",
        "park": "Campground",
    },
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()

print("Lead time (days) by campground — selected quantiles:")
print(
    df_summer.groupby("park", observed=True)["lead_time_days"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
    .to_string()
)


Lead time (days) by campground — selected quantiles:
                                       count     median       mean
park                                                              
Lewis Lake Campground                   4813  79.319994  82.059600
Indian Creek Campground (Yellowstone)   4186  40.010051  82.242311
Slough Creek Campground                  787  13.414473  54.831482
Mammoth Campground (Yellowstone)        5229  13.295574  22.618487


**Interpretation:** The **box plots** make the ordering of “how far in advance” people book visible beyond a single average. **Lewis Lake** sits at a **much higher** central tendency and upper range than the other three—consistent with guests booking that campground **many more days ahead** on typical stays. **Mammoth** and **Slough Creek** show **tighter, shorter** lead-time distributions (people book closer to the trip). **Indian Creek** is **in between**, suggesting middling planning horizons. Longer lead times can signal **high expected scarcity or popularity**, but they can also reflect **release timing, inventory rules, or trip-type differences**, so desirability claims should be paired with capacity and booking-window policy checks.

---

## Section 4 — Visualization Rationale

Section 3 built three Plotly figures for Yellowstone summer 2024 camping reservations; the same views are exported as SVG in the course **`Week 6`** folder. Below, each figure is shown in order, followed by a short rationale: **why this chart type** for the data structure, and **what the reader should take away**.

In [1]:
import re
from pathlib import Path

from IPython.display import HTML, Markdown, display

# Section 4 uses the exported MP1 charts in the course Week 6 folder:
#   mp1_q1_cancellation_proxy_vs_lead_time.svg
#   mp1_q2_ridb_campsites_vs_avg_stay.svg
#   mp1_q3_booking_lead_time_boxplot.svg
WEEK6_SVG_NAMES = (
    "mp1_q1_cancellation_proxy_vs_lead_time.svg",
    "mp1_q2_ridb_campsites_vs_avg_stay.svg",
    "mp1_q3_booking_lead_time_boxplot.svg",
)

_FIT_STYLE = (
    "max-width:100% !important;width:100% !important;height:auto !important;"
    "display:block !important;box-sizing:border-box !important;"
)


def _svg_fit_screen(svg: str, *, chart_id: str) -> str:
    """Scale Plotly/Kaleido SVGs to the notebook output width (strip huge root width/height)."""
    m = re.search(r"(?is)<svg([^>]*)>", svg)
    if not m:
        return f'<div class="mp1-s4-wrap" style="width:100%;overflow-x:auto;">{svg}</div>'
    attrs = m.group(1)
    attrs = re.sub(r'(?i)\s+width="[^"]*"', "", attrs)
    attrs = re.sub(r'(?i)\s+height="[^"]*"', "", attrs)
    if not re.search(r"(?i)\bpreserveAspectRatio\s*=", attrs):
        attrs = ' preserveAspectRatio="xMidYMid meet"' + attrs
    if re.search(r'(?i)style\s*=\s*""', attrs):
        attrs = re.sub(r'(?i)style\s*=\s*""', f'style="{_FIT_STYLE}"', attrs, count=1)
    elif re.search(r'(?i)style\s*=\s*"[^"]*"', attrs):
        attrs = re.sub(
            r'(?i)(style\s*=\s*")([^"]*)(")',
            lambda mm: f'{mm.group(1)}{mm.group(2)};{_FIT_STYLE}{mm.group(3)}',
            attrs,
            count=1,
        )
    else:
        attrs = f' style="{_FIT_STYLE}"' + attrs
    fixed = "<svg" + attrs + ">" + svg[m.end() :]
    return (
        f'<div id="{chart_id}" class="mp1-s4-wrap" style="width:100%;max-width:100%;'
        "margin:0 auto 0.75rem auto;overflow-x:auto;box-sizing:border-box;line-height:0;"
        f'">{fixed}</div>'
    )


def week6_dir() -> Path:
    """Week 6/ sits next to MP1_yellowstone/ under the HCDE 530 course root."""
    cwd = Path.cwd().resolve()
    for base in (cwd, cwd.parent):
        d = base / "Week 6"
        if all((d / name).is_file() for name in WEEK6_SVG_NAMES):
            return d
    raise FileNotFoundError(
        "Expected these SVGs under Week 6/:\n  "
        + "\n  ".join(WEEK6_SVG_NAMES)
        + "\n(Run Week 6/export_mp1_yellowstone_charts_svg.py if they are missing.)"
    )


W6 = week6_dir()

charts = [
    (
        "Question 1 — Cancellation proxy rate vs. median booking lead time",
        W6 / "mp1_q1_cancellation_proxy_vs_lead_time.svg",
        """**Why this chart type:** A **scatter** fits because each campground is one point in a plane defined by two **aggregated numeric** summaries—cancellation *proxy* rate (share of reservations with `$0` total paid) and **median** booking lead time (days from order to trip start). Marker **size** carries reservation count so a high rate on few stays does not read the same as the same rate on many reservations.

**Reader takeaway:** Compare campgrounds along both “more proxy cancellations” and “booking further in advance,” and use point size to judge how stable those summaries are across volume.""",
    ),
    (
        "Question 2 — RIDB campsite inventory vs. mean stay length",
        W6 / "mp1_q2_ridb_campsites_vs_avg_stay.svg",
        """**Why this chart type:** A **bar** chart fits because the main signal is a **nonnegative magnitude per campground** (RIDB campsite record counts), where bar length supports direct comparison and sort order highlights scale. **Color** encodes a second numeric—mean nights parsed from reservation text—so *listed inventory* and *observed stay length* stay separate channels instead of being forced onto one axis.

**Reader takeaway:** Relate official campsite listings to typical stay behavior: ask whether larger RIDB counts line up with longer mean stays, or whether high inventory can sit next to shorter means.""",
    ),
    (
        "Question 3 — Booking lead time distribution by campground",
        W6 / "mp1_q3_booking_lead_time_boxplot.svg",
        """**Why this chart type:** A **box** plot fits because lead time is **continuous at the reservation level**, but the question is **comparative across categorical campgrounds**; boxes summarize median, quartiles, and outliers without drawing one point per reservation on the category axis.

**Reader takeaway:** Judge how far in advance people book each facility—including spread and extremes—not only a single campground average.""",
    ),
]

for idx, (title, svg_path, blurb) in enumerate(charts):
    if not svg_path.is_file():
        raise FileNotFoundError(svg_path)
    display(Markdown(f"### {title}"))
    raw = svg_path.read_text(encoding="utf-8")
    display(HTML(_svg_fit_screen(raw, chart_id=f"mp1-s4-{idx}")))
    display(Markdown(blurb))


### Question 1 — Cancellation proxy rate vs. median booking lead time

**Why this chart type:** A **scatter** fits because each campground is one point in a plane defined by two **aggregated numeric** summaries—cancellation *proxy* rate (share of reservations with `$0` total paid) and **median** booking lead time (days from order to trip start). Marker **size** carries reservation count so a high rate on few stays does not read the same as the same rate on many reservations.

**Reader takeaway:** Compare campgrounds along both “more proxy cancellations” and “booking further in advance,” and use point size to judge how stable those summaries are across volume.

### Question 2 — RIDB campsite inventory vs. mean stay length

**Why this chart type:** A **bar** chart fits because the main signal is a **nonnegative magnitude per campground** (RIDB campsite record counts), where bar length supports direct comparison and sort order highlights scale. **Color** encodes a second numeric—mean nights parsed from reservation text—so *listed inventory* and *observed stay length* stay separate channels instead of being forced onto one axis.

**Reader takeaway:** Relate official campsite listings to typical stay behavior: ask whether larger RIDB counts line up with longer mean stays, or whether high inventory can sit next to shorter means.

### Question 3 — Booking lead time distribution by campground

**Why this chart type:** A **box** plot fits because lead time is **continuous at the reservation level**, but the question is **comparative across categorical campgrounds**; boxes summarize median, quartiles, and outliers without drawing one point per reservation on the category axis.

**Reader takeaway:** Judge how far in advance people book each facility—including spread and extremes—not only a single campground average.

---

## Section 5 — Conclusions

Briefly answer the reflection prompts below. A drafted response is in the **next cell**, organized with subheadings so you can skim or edit in place.

**Reflection prompts**

1. What is the most important thing your analysis revealed?
2. What surprised you?
3. What would you investigate next if you had more time or data?
4. What are the limitations of this analysis — what can’t you conclude from this data?

Then complete the **Competency Claim** (same cell as the summary, below the horizontal rule).

### Summary of findings

#### 1. Most important finding

Across this FY24 **summer Yellowstone** extract, the four front-country campgrounds **do not behave as one unit**. By `park`, the patterns shift for:

- **Cancellation proxy** — share of reservations with `$0` `totalpaid` (a documented *proxy* only, not a true cancel flag)
- **Median booking lead time** — days from `orderdate` to `startdate`
- **RIDB inventory** — campsite record counts from the API
- **Stay length** — mean nights parsed from the `nights` text field
- **Lead-time spread** — full distribution of advance booking, not just an average

**Lewis Lake** stands out in the exploratory plots as **booked further ahead**, with a **longer right tail** in lead time, compared with Mammoth or Slough Creek.

---

#### 2. What surprised me

**High RIDB campsite record counts did not map neatly onto longer mean stays** when read as a simple “more sites → longer trips” story. That cautions against treating **catalog inventory** as a direct stand-in for **observed reservation behavior** from `nights` alone.

---

#### 3. What I would investigate next

With more time or richer data I would:

- Add **true cancellation or refund metadata** (instead of relying only on the `$0` paid proxy)
- Bring in **site type, vehicle length limits, and capacity** to explain demand and lead times
- Extend beyond **June–August** to full seasons and year-over-year trends
- Align **RIDB pull dates** (or snapshots) with **reservation order dates** so inventory and bookings are temporally comparable

---

#### 4. Limitations

This notebook **cannot**:

- **Prove cancellations** from `$0` total paid — that field can reflect comps, transfers, processing quirks, or other non-cancel reasons
- **Explain why** patterns appear — the work is **descriptive**, not causal
- **Generalize** beyond this **processed slice** and these **four facilities**

Claims about popularity, fairness, or park policy would need **additional evidence** outside this extract.

---

## Competency Claim

In an `mp1.md` file in your GitHub repository, write a **short competency claim** (2–4 sentences) for **each domain** your project actually demonstrates. Point to something concrete you did in this notebook (e.g., a cleaning step, a `groupby` analysis, a chart, or a limitation you named).

| Domain | Theme |
|--------|--------|
| **C3** | Data cleaning and file handling |
| **C5** | Data analysis with pandas |
| **C6** | Data visualization |
| **C7** | Critical evaluation and professional judgment |

You do **not** need to claim every domain — only the ones your work supports.
